# Building a chatbot finetune on Qwen/Qwen3-4B

In [1]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')

!git clone https://{token}@github.com/nando0307/ChatWithMe.git
%cd ChatWithMe
!git config user.name "nando"
!git config user.email "lenando0307@gmail.com"

Cloning into 'ChatWithMe'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 28 (delta 3), reused 23 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 29.82 KiB | 1.36 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/ChatWithMe


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [25]:
!cp "/content/drive/MyDrive/Colab Notebooks/ChatWithMe.ipynb" /content/ChatWithMe/

In [26]:
import nbformat

with open("/content/ChatWithMe/ChatWithMe.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open("/content/ChatWithMe/ChatWithMe.ipynb", "w") as f:
    nbformat.write(nb, f)

In [27]:
!cd /content/ChatWithMe && git add . && git commit -m "" && git push

[main d3c4198] add model initialization(bnb config, lora config)
 1 file changed, 937 insertions(+), 1 deletion(-)
 rewrite ChatWithMe.ipynb (99%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 8.82 KiB | 8.82 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/nando0307/ChatWithMe.git
   d2c7264..d3c4198  main -> main


## 1. Install and Import

In [3]:
!pip install -q accelerate peft bitsandbytes "transformers>=4.51.0" trl datasets matplotlib

import torch
import math
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.5 MB/s eta 0:00:00


## 2. Data Preprocessing
Load the cleaned Alpaca dataset, format each example into ChatML template for Qwen3 compatibility, and split into 90/10 train/validation sets.

In [4]:
# Load dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(f'Total examples: {len(dataset)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Total examples: 51760


In [5]:
print(dataset[0])

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [6]:
# Format into ChatML
def format_chatml(example):
    instruction = example["instruction"]
    if example["input"]:
        instruction = f"{instruction}\n{example['input']}"

    text = (
        f"<|im_start|>user\n{instruction}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": text}

In [7]:
dataset = dataset.map(format_chatml, remove_columns=dataset.column_names)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [8]:
# Split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\nSample:\n{train_dataset[0]['text']}")

Train: 46584 | Val: 5176

Sample:
<|im_start|>user
Analyze the user's writing style in the given text snippet and provide 3 suggestions to improve it.
Weather now bad outside also it is cold. People no have fun time, always grumble because of cold weather. Use warm clothes protect from freezing.<|im_end|>
<|im_start|>assistant
1. Improve grammar and sentence structure: The sentences are fragmented and lack proper grammar. Consider rewriting them as, 'The weather outside is bad, and it's cold. People are not having a good time, and they are always grumbling because of the cold weather. Wear warm clothes to protect yourself from freezing.' 
2. Use more descriptive words: Replace 'bad' and 'cold' with more specific and descriptive words like 'gloomy' or 'frigid.'
3. Make the text flow smoothly by using conjunctions and transitions: Use words like 'however,' 'moreover,' and 'therefore' to connect ideas and make the text easier to read.<|im_end|>


## 3. Model Initialization
Load Qwen3-4B with 4-bit quantization, configure the tokenizer, and apply LoRA adapters and MLP projection layers



In [9]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # load the model weights in 4-bit instead of default 16/32-bit -> fit in smaller VRAM
    bnb_4bit_quant_type="nf4", # uses NormalFloat 4 quantizaton (designed for nn weights) -> nf4 is optimized for normal distribution > 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16, # when doing the math (matrix) -> uses the FP 16 for more accurate calculation
    bnb_4bit_use_double_quant=True, # quantize the weights will also create the scaling factor to decode them back, so that also quantize the scaling factor will x2 the compress
)

In [10]:
# Load model
model_name = "Qwen/Qwen3-4B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", # auto figure out where to put the model on GPU/CPU
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # pad the shorter sequences to match the longest one
tokenizer.padding_side = "right" # padding on the right of the sequence

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [11]:
# LoRA config
lora_config = LoraConfig(
    r=8, # r means rank, lower the rank means less matrix mutiplication to do -> less memory require
    lora_alpha=16, # the scaling factor, avoid the inflate when increase the rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [12]:
# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089


## 4. Model Training
Configure SFTTrainer with LoRA, cosine learning rate scheduler, and FP16 mixed precision. Train for 1 epoch on the formatted ChatML dataset and save the adapter weights.

In [15]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/Colab Notebooks/ChatWithMe_results",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    lr_scheduler_type="cosine", # using scheduler cosine
    warmup_steps=575, #lr ramps up from 0 to 2e-4 -> prevent unstable updates at the beginner of the model
    fp16=False, # use the floating point 16 bit for training computations
    bf16=True,
    eval_strategy="steps", # run the evaluation every 200 steps
    eval_steps=200,
    save_strategy="steps", # checkpoint to drive that can resume if crashe occur
    save_steps=200,
    logging_steps=50, # print the training loss every 50 steps -> visualization
    report_to="none",
    save_total_limit=2, # keep the 2 most recent checkpoints, delete the old one to save memory
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
aa

In [16]:
trainer.train()

# Save the LoRA adapter
trainer.model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/qwen3-4b-alpaca-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/qwen3-4b-alpaca-lora")
print("Adapter saved to qwen3-4b-alpaca-lora/")

Step,Training Loss,Validation Loss
200,1.138221,1.141741
400,1.122206,1.123913
600,1.109000,1.116651
800,1.096498,1.108388
1000,1.100028,1.102731
1200,1.085982,1.098384
1400,1.087542,1.097587


Adapter saved to qwen3-4b-alpaca-lora/


## 5. Visualization of Metrics and Loss

## 6. Model Evaluation

## 7.  Model Inference

## 8. Merge & Push to HuggingFace Hub